In [78]:
import pandas as pd

caminho_arquivo = "Dados Tratados .xlsx" 
df = pd.read_excel(caminho_arquivo)

# Ação aplicada na tabela (df) e não na biblioteca (pd)
df.head()

,Estado,Taxa de MVI,Renda,Gini,Desemprego Médio 2024
0,Acre,20.326334,1271,0.502,7.700
1,Alagoas,35.433638,1331,0.516,8.500
2,Amapá,45.090099,1514,0.509,9.200
3,Amazonas,27.398803,1238,0.474,8.575
4,Bahia,40.645061,1366,0.477,11.175


In [79]:
from sklearn.preprocessing import MinMaxScaler

normalizador = MinMaxScaler()
colunas_alvo = ['Taxa de MVI','Renda','Gini', 'Desemprego Médio 2024']

df[colunas_alvo] = normalizador.fit_transform(df[colunas_alvo])

df.head()



,Estado,Taxa de MVI,Renda,Gini,Desemprego Médio 2024
0,Acre,0.329459,0.081960,0.615385,0.574018
1,Alagoas,0.738527,0.107309,0.735043,0.670695
2,Amapá,1.000000,0.184622,0.675214,0.755287
3,Amazonas,0.520963,0.068019,0.376068,0.679758
4,Bahia,0.879639,0.122095,0.401709,0.993958


In [ ]:
import pandas as pd

# 1. Limpeza de segurança (Nomes das colunas)
df.columns = df.columns.str.strip()
if 'Desmprego Médio 2024' in df.columns:
    df = df.rename(columns={'Desmprego Médio 2024': 'Desemprego Médio 2024'})

# 2. Cálculo do Índice Composto Linear (Substituindo o K-Means)
peso = 0.25
df['Score Desenvolvimento'] = (
    peso * df['Renda'] +
    peso * (1 - df['Gini']) +
    peso * (1 - df['Desemprego Médio 2024']) +
    peso * (1 - df['Taxa de MVI'])
)

# 3. Classificação Absoluta em Régua Fixa (A sua lógica)
# Definimos os cortes exatamente como você pediu: 0 a 0.3333 | 0.3333 a 0.6667 | 0.6667 a 1.0
df['Perfil Socioeconomico'] = pd.cut(
    df['Score Desenvolvimento'], 
    bins=[-0.01, 0.3333, 0.6667, 1.01], # Usamos -0.01 e 1.01 nas pontas para garantir que notas extremas não fiquem de fora por arredondamento
    labels=['Vulnerável', 'Intermediário', 'Desenvolvido']
)

df.head(27)

,Estado,Taxa de MVI,Renda,Gini,Desemprego Médio 2024,Score Desenvolvimento,Perfil Socioeconomico
0,Acre,0.329459,0.081960,0.615385,0.574018,0.390775,Intermediário
1,Alagoas,0.738527,0.107309,0.735043,0.670695,0.240761,Vulnerável
2,Amapá,1.000000,0.184622,0.675214,0.755287,0.188530,Vulnerável
3,Amazonas,0.520963,0.068019,0.376068,0.679758,0.372807,Intermediário
4,Bahia,0.879639,0.122095,0.401709,0.993958,0.211697,Vulnerável
5,Ceará,0.795763,0.062526,0.487179,0.531722,0.311965,Vulnerável
6,Distrito Federal,0.020542,1.000000,1.000000,0.770393,0.552266,Intermediário
7,Espírito Santo,0.425955,0.436840,0.427350,0.202417,0.595279,Intermediário
8,Goiás,0.287064,0.431348,0.273504,0.290030,0.645187,Intermediário
9,Maranhão,0.601328,0.000000,0.487179,0.558912,0.338145,Intermediário


In [82]:
matriz_correlacao = df[colunas_alvo].corr(method='pearson')

matriz_correlacao

,Taxa de MVI,Renda,Gini,Desemprego Médio 2024
Taxa de MVI,1.000000,-0.688277,0.186983,0.454806
Renda,-0.688277,1.000000,-0.130724,-0.394881
Gini,0.186983,-0.130724,1.000000,0.776640
Desemprego Médio 2024,0.454806,-0.394881,0.776640,1.000000


In [83]:
nome_arquivo_final = "base_mvi_precessada.xlsx"
df.to_excel(nome_arquivo_final,index=False)
print('Base de dados importada!')

Base de dados importada!


In [84]:
import sqlite3

conexao = sqlite3.connect("banco_seguranca_publica.db")
df.to_sql('tb_indicadores_estaduais', conexao, if_exists="replace", index=False)
conexao.close()

print("ETL Finalizado: Índice Linear calculado e Banco de Dados atualizado!")

ETL Finalizado: Índice Linear calculado e Banco de Dados atualizado!
